# CREDIT RISK PREDICTION — MEMBER 3: LIGHTGBM FROM SCRATCH (PURE NUMPY)

**CS114 · Home Credit Default Risk · Tự cài đặt chi tiết thuật toán LightGBM bằng tay (Pure NumPy)**

Notebook này tuân thủ tuyệt đối **4 quy định dùng chung** của nhóm trong `README.md`:
1. **Dùng chung pipeline dữ liệu:** Nạp dữ liệu qua `src.data_loader.get_train_test_data(scale=False)` — không tự đọc CSV, không tự chia lại Train/Test, không chỉnh sửa bất kỳ file nào trong `src/`.
2. **Dùng chung bộ đánh giá:** Chấm điểm và xuất kết quả bằng `src.metrics.evaluate_and_save()` ra đúng file `results/member3_lightgbm.json`.
3. **Tự code thuật toán LightGBM chi tiết bằng tay (Pure NumPy)** theo đúng bài báo gốc *Ke et al., "LightGBM: A Highly Efficient Gradient Boosting Decision Tree" (NeurIPS 2017)* với đầy đủ **5 trụ cột cốt lõi**:
   - **Second-Order Gradient Boosting (Newton-Raphson)** cho Binary Log-Loss có `scale_pos_weight` và điều chuẩn L1/L2 ($\alpha, \lambda$).
   - **Histogram-Based Split Finding** (lượng tử hóa `uint8` theo quantile) kết hợp **Histogram Subtraction Trick** ($H_{\text{large}} = H_{\text{parent}} - H_{\text{small}}$).
   - **Leaf-Wise (Best-First) Tree Growth** bằng Priority Queue (`heapq`) kèm kiểm soát `num_leaves`, `max_depth`, `min_child_samples`, `min_child_weight`.
   - **GOSS (Gradient-based One-Side Sampling)** giữ top $a\%$ mẫu có $|g_i|$ lớn nhất, lấy mẫu $b\%$ phần còn lại và nhân trọng số bù $\frac{1-a}{b}$.
   - **EFB (Exclusive Feature Bundling)** gom các đặc trưng thưa xung khắc nhau bằng Greedy Graph Coloring và Bin Offset Merging.
4. **Kiểm chứng toàn diện:** Thực hiện Ablation Study cho từng kỹ thuật + đối chiếu công bằng **From Scratch (Pure NumPy)** vs **Library (`lightgbm.LGBMClassifier`)**.

In [ ]:
import heapq
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
try:
    from IPython.display import display
except ImportError:
    display = print

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
)

# Import module dùng chung của nhóm (KHÔNG sửa các file trong src/)
from src.config import DATA_DIR, RANDOM_STATE
from src.data_loader import get_train_test_data
from src.metrics import evaluate_and_save, plot_confusion_matrix

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
np.random.seed(RANDOM_STATE)

# Kiểm tra đủ 6 file CSV bắt buộc trong data/
REQUIRED_CSV = [
    "application_train.csv",
    "bureau.csv",
    "previous_application.csv",
    "installments_payments.csv",
    "POS_CASH_balance.csv",
    "credit_card_balance.csv",
]

file_status = pd.DataFrame([
    {
        "file": name,
        "exists": (DATA_DIR / name).is_file(),
        "size_MiB": round((DATA_DIR / name).stat().st_size / (1024 ** 2), 1)
        if (DATA_DIR / name).is_file() else None,
    }
    for name in REQUIRED_CSV
])
display(file_status)

missing = file_status.loc[~file_status["exists"], "file"].tolist()
if missing:
    raise FileNotFoundError(
        "Thiếu file CSV Home Credit trong thư mục `data/`: "
        + ", ".join(missing)
        + "\nHãy tải từ https://www.kaggle.com/competitions/home-credit-default-risk/data và đặt vào `data/`."
    )
print("[OK] Đã tìm thấy đầy đủ 6 file CSV Home Credit trong data/!")

## 1. Nạp dữ liệu dùng chung của nhóm (`src/data_loader.py`)

- Gọi `get_train_test_data(scale=False)` để nhận đúng bộ Train/Test (`75% / 25%`, `random_state=1`, `stratify=y`) cùng ~122 đặc trưng đã qua Top-1 Kaggle Feature Engineering + `neighbors_target_mean_500` + Ridge Feature Selection.
- Truyền `scale=False` vì mô hình cây quyết định bất biến với phép biến đổi đơn điệu — giữ thang đo gốc giúp bước **Histogram Binning** và **EFB (phát hiện đặc trưng thưa bằng 0)** hoạt động chính xác hơn.
- Tách thêm một tập **Validation nội bộ (15% từ `X_train`)** để chạy **Early Stopping** và **Threshold Tuning** — tuyệt đối **không chạm vào `X_test`** trong quá trình huấn luyện và chọn ngưỡng.

In [ ]:
t_load = time.perf_counter()
X_train, X_test, y_train, y_test, feature_names = get_train_test_data(scale=False)
print(f"Thời gian nạp & xử lý dữ liệu dùng chung: {time.perf_counter() - t_load:.2f}s")

y_train = np.asarray(y_train, dtype=np.int8)
y_test = np.asarray(y_test, dtype=np.int8)

# Kiểm tra tính toàn vẹn dữ liệu
assert X_train.ndim == 2 and X_test.ndim == 2
assert X_train.shape[1] == X_test.shape[1] == len(feature_names)
assert np.isfinite(X_train).all() and np.isfinite(X_test).all(), "Dữ liệu còn chứa NaN/Inf!"

# Tách tập Validation nội bộ từ X_train (15%) cho Early Stopping & Threshold Tuning (zero test leakage)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.15,
    random_state=RANDOM_STATE,
    stratify=y_train,
)

# Tính trọng số cân bằng lớp (scale_pos_weight = neg / pos) trên tập train nội bộ
n_neg = int((y_tr == 0).sum())
n_pos = int((y_tr == 1).sum())
SCALE_POS_WEIGHT = float(n_neg / max(n_pos, 1))

summary_df = pd.DataFrame([
    {"Split": "Train (nội bộ - 85%)", "Samples": len(y_tr), "High-Risk (1)": int(y_tr.sum()), "Positive Rate": f"{y_tr.mean()*100:.2f}%"},
    {"Split": "Val   (nội bộ - 15%)", "Samples": len(y_val), "High-Risk (1)": int(y_val.sum()), "Positive Rate": f"{y_val.mean()*100:.2f}%"},
    {"Split": "Test  (Hold-out nhóm)", "Samples": len(y_test), "High-Risk (1)": int(y_test.sum()), "Positive Rate": f"{y_test.mean()*100:.2f}%"},
])
display(summary_df)
print(f"Số đặc trưng đầu vào: {len(feature_names)} | scale_pos_weight = {SCALE_POS_WEIGHT:.4f}")

## 2. Cơ sở toán học & Kiến trúc thuật toán LightGBM (Tự cài đặt từ đầu)

### 2.1. Second-Order Gradient Boosting cho Binary Log-Loss có trọng số lớp
Với mỗi mẫu $i$ có nhãn $y_i \in \{0, 1\}$ và trọng số lớp $w_i = \text{scale\_pos\_weight}$ nếu $y_i = 1$ (ngược lại $w_i = 1$), tại vòng lặp boosting thứ $t$, xác suất dự đoán hiện tại là $\hat{p}_i = \sigma(F_{t-1}(x_i)) = \frac{1}{1 + e^{-F_{t-1}(x_i)}}$.

Khai triển Taylor bậc 2 của hàm mất mát **Weighted Binary Cross-Entropy**:
$$\mathcal{L}^{(t)} \approx \sum_{i=1}^{N} \left[ g_i f_t(x_i) + \frac{1}{2} h_i f_t(x_i)^2 \right] + \Omega(f_t)$$

trong đó Gradient bậc 1 và Hessian bậc 2 được tính giải tích:
$$g_i = w_i (\hat{p}_i - y_i), \qquad h_i = w_i \hat{p}_i (1 - \hat{p}_i)$$

Với điều chuẩn kết hợp **L1 ($\alpha$)** và **L2 ($\lambda$)**, đặt hàm ngưỡng mềm $S(G, \alpha) = \text{sign}(G)\max(|G| - \alpha, 0)$, giá trị tối ưu tại lá $j$ là:
$$v_j^* = -\frac{S(G_j, \alpha)}{H_j + \lambda}, \qquad G_j = \sum_{i \in I_j} g_i, \quad H_j = \sum_{i \in I_j} h_i$$

Độ lợi (**Split Gain**) khi tách một nút thành con trái $L$ và con phải $R$ (với ngưỡng cắt phạt $\gamma$):
$$\text{Gain} = \frac{1}{2}\left[\frac{S(G_L, \alpha)^2}{H_L + \lambda} + \frac{S(G_R, \alpha)^2}{H_R + \lambda} - \frac{S(G_L + G_R, \alpha)^2}{H_L + H_R + \lambda}\right] - \gamma$$

### 2.2. Bốn cải tiến tạo nên tốc độ & độ chính xác của LightGBM
1. **Histogram-Based Split Finding:** Rời rạc hóa toàn bộ giá trị thực của mỗi đặc trưng vào tối đa `max_bins` thùng (`uint8`). Thay vì sắp xếp $O(N \log N)$ ở mỗi nút, ta cộng dồn $G, H$ vào mảng `max_bins` bằng `np.bincount` trong $O(N)$, rồi quét tìm điểm tách tối ưu trong $O(\text{max\_bins})$.
2. **Histogram Subtraction Trick:** Khi tách một nút cha thành 2 nút con, **chỉ dựng histogram cho nút con có ít mẫu hơn**; histogram của nút con còn lại tính trực tiếp bằng phép trừ mảng:
   $$\text{Hist}_{\text{large}} = \text{Hist}_{\text{parent}} - \text{Hist}_{\text{small}}$$
   giúp giảm hơn **50%** khối lượng tính toán mà kết quả số học **chính xác tuyệt đối** (sai số $= 0$).
3. **Leaf-Wise (Best-First) Growth:** Sử dụng **Max-Heap (`heapq`)** để luôn chọn tách chiếc lá mang lại $\text{Gain}$ lớn nhất toàn cây cho tới khi đạt `num_leaves` (khác với cách tách đồng đều từng tầng *level-wise*), giúp giảm hàm mất mát nhanh hơn với cùng số lá.
4. **GOSS & EFB:**
   - **GOSS (Gradient-based One-Side Sampling):** Giữ lại top $a\%$ mẫu có $|g_i|$ lớn nhất (mẫu khó đoán, đóng góp lớn vào Gain), lấy mẫu ngẫu nhiên $b\%$ trong nhóm $|g_i|$ nhỏ và nhân trọng số bù $\frac{1 - a}{b}$ lên $g_i, h_i$ để ước lượng Gain không chệch.
   - **EFB (Exclusive Feature Bundling):** Dùng thuật toán tô màu đồ thị tham lam (*Greedy Graph Coloring*) gộp các đặc trưng thưa ít khi đồng thời khác giá trị mặc định vào chung một bundle bằng độ lệch bin (*bin offset*), giảm số chiều duyệt histogram.

In [ ]:
class HistogramBinner:
    """Bước 1 của LightGBM: Lượng tử hóa (Quantile Binning) đặc trưng liên tục về uint8."""

    def __init__(self, max_bins: int = 63, subsample_for_bin: int = 100_000, random_state: int = 1):
        self.max_bins = max_bins
        self.subsample_for_bin = subsample_for_bin
        self.random_state = random_state
        self.bin_edges_ = []
        self.n_bins_per_feature_ = None

    def fit(self, X: np.ndarray):
        X = np.asarray(X, dtype=np.float32)
        n_samples, n_features = X.shape
        self.bin_edges_ = []
        n_bins_list = []

        # Lấy mẫu con để tìm ngưỡng quantile nhanh mà vẫn chính xác phân phối
        if n_samples > self.subsample_for_bin:
            rng = np.random.default_rng(self.random_state)
            idx = rng.choice(n_samples, size=self.subsample_for_bin, replace=False)
            X_sample = X[idx]
        else:
            X_sample = X

        quantiles = np.linspace(0.0, 1.0, self.max_bins + 1)[1:-1]
        for j in range(n_features):
            col = X_sample[:, j]
            uniq = np.unique(col)
            if len(uniq) <= self.max_bins:
                if len(uniq) <= 1:
                    edges = np.array([], dtype=np.float32)
                else:
                    # Trung điểm giữa các giá trị phân biệt liên tiếp
                    edges = (0.5 * (uniq[:-1] + uniq[1:])).astype(np.float32)
            else:
                edges = np.unique(np.quantile(col, quantiles)).astype(np.float32)
            self.bin_edges_.append(edges)
            n_bins_list.append(len(edges) + 1)

        self.n_bins_per_feature_ = np.array(n_bins_list, dtype=np.int32)
        return self

    def transform(self, X: np.ndarray) -> np.ndarray:
        X = np.asarray(X, dtype=np.float32)
        n_samples, n_features = X.shape
        X_binned = np.empty((n_samples, n_features), dtype=np.uint8)
        for j in range(n_features):
            edges = self.bin_edges_[j]
            if len(edges) == 0:
                X_binned[:, j] = 0
            else:
                X_binned[:, j] = np.searchsorted(edges, X[:, j], side="right").astype(np.uint8)
        return X_binned

    def fit_transform(self, X: np.ndarray) -> np.ndarray:
        return self.fit(X).transform(X)


class ExclusiveFeatureBundler:
    """Bước 2 của LightGBM: EFB — Exclusive Feature Bundling (Algorithm 3 & 4, Ke et al., 2017)."""

    def __init__(
        self,
        sparse_threshold: float = 0.85,
        max_conflict_rate: float = 0.001,
        max_bundle_bins: int = 255,
        enabled: bool = True,
    ):
        self.sparse_threshold = sparse_threshold
        self.max_conflict_rate = max_conflict_rate
        self.max_bundle_bins = max_bundle_bins
        self.enabled = enabled

        self.n_orig_features_ = 0
        self.n_bundled_features_ = 0
        self.bundles_ = []
        self.bundle_offsets_ = []
        self.default_bins_ = None
        self.bundle_n_bins_ = None
        self.bin_to_orig_feature_ = []

    def fit(self, X_binned: np.ndarray, n_bins_per_feature: np.ndarray):
        n_samples, n_features = X_binned.shape
        self.n_orig_features_ = n_features
        self.default_bins_ = np.zeros(n_features, dtype=np.uint8)

        if not self.enabled:
            self.bundles_ = [[j] for j in range(n_features)]
            self.bundle_offsets_ = [[0] for _ in range(n_features)]
            self.bundle_n_bins_ = n_bins_per_feature.copy()
            self.bin_to_orig_feature_ = [
                np.full(int(n_bins_per_feature[j]), j, dtype=np.int32) for j in range(n_features)
            ]
            self.n_bundled_features_ = n_features
            return self

        # 1. Phân loại đặc trưng thưa (tần suất bin phổ biến nhất >= sparse_threshold) vs dày
        sparse_feats = []
        dense_feats = []
        for j in range(n_features):
            nb = int(n_bins_per_feature[j])
            if nb <= 1:
                dense_feats.append(j)
                continue
            counts = np.bincount(X_binned[:, j], minlength=nb)
            mode_bin = int(np.argmax(counts))
            if counts[mode_bin] / n_samples >= self.sparse_threshold:
                self.default_bins_[j] = mode_bin
                sparse_feats.append(j)
            else:
                dense_feats.append(j)

        # 2. Algorithm 3 — Greedy Bundling dựa trên đồ thị xung đột của các đặc trưng thưa
        sparse_bundles = []
        if len(sparse_feats) > 1:
            nz_mat = np.empty((n_samples, len(sparse_feats)), dtype=np.float32)
            for idx_s, j in enumerate(sparse_feats):
                nz_mat[:, idx_s] = (X_binned[:, j] != self.default_bins_[j]).astype(np.float32)
            # Tính số lần đồng thời khác default_bin bằng nhân ma trận BLAS
            conflict_mat = (nz_mat.T @ nz_mat).astype(np.int64)
            np.fill_diagonal(conflict_mat, 0)

            max_conflicts = int(self.max_conflict_rate * n_samples)
            degrees = (conflict_mat > 0).sum(axis=1)
            search_order = np.argsort(-degrees)

            bundle_bins = []
            for local_i in search_order:
                orig_f = sparse_feats[local_i]
                f_nondefault = int(n_bins_per_feature[orig_f]) - 1
                placed = False
                for b_idx, bundle in enumerate(sparse_bundles):
                    if bundle_bins[b_idx] + f_nondefault > self.max_bundle_bins:
                        continue
                    total_conf = sum(conflict_mat[local_i, local_m] for local_m in bundle)
                    if total_conf <= max_conflicts:
                        bundle.append(local_i)
                        bundle_bins[b_idx] += f_nondefault
                        placed = True
                        break
                if not placed:
                    sparse_bundles.append([local_i])
                    bundle_bins.append(1 + f_nondefault)  # bin 0 dành cho trạng thái all-default

            sparse_bundles = [[sparse_feats[i] for i in b] for b in sparse_bundles]
        elif len(sparse_feats) == 1:
            sparse_bundles = [[sparse_feats[0]]]

        # 3. Algorithm 4 — Merge Exclusive Features bằng Bin Offsets
        self.bundles_ = [[j] for j in dense_feats] + sparse_bundles
        self.bundle_offsets_ = []
        bundle_n_bins = []
        self.bin_to_orig_feature_ = []

        for bundle in self.bundles_:
            if len(bundle) == 1:
                j = bundle[0]
                nb = int(n_bins_per_feature[j])
                self.bundle_offsets_.append([0])
                bundle_n_bins.append(nb)
                self.bin_to_orig_feature_.append(np.full(nb, j, dtype=np.int32))
            else:
                offsets = []
                total_bins = 1  # bin 0 = tất cả đặc trưng trong bundle đều ở default_bin
                bin_owner = [bundle[0]]
                for j in bundle:
                    offsets.append(total_bins)
                    nondefault_nb = int(n_bins_per_feature[j]) - 1
                    bin_owner.extend([j] * nondefault_nb)
                    total_bins += nondefault_nb
                self.bundle_offsets_.append(offsets)
                bundle_n_bins.append(total_bins)
                self.bin_to_orig_feature_.append(np.array(bin_owner, dtype=np.int32))

        self.bundle_n_bins_ = np.array(bundle_n_bins, dtype=np.int32)
        self.n_bundled_features_ = len(self.bundles_)
        return self

    def transform(self, X_binned: np.ndarray) -> np.ndarray:
        n_samples = X_binned.shape[0]
        X_out = np.zeros((n_samples, self.n_bundled_features_), dtype=np.uint8)
        for b_idx, (bundle, offsets) in enumerate(zip(self.bundles_, self.bundle_offsets_)):
            if len(bundle) == 1:
                X_out[:, b_idx] = X_binned[:, bundle[0]]
            else:
                col_out = np.zeros(n_samples, dtype=np.uint8)
                for j, off in zip(bundle, offsets):
                    col = X_binned[:, j]
                    d_bin = self.default_bins_[j]
                    mask = col != d_bin
                    if np.any(mask):
                        vals = col[mask]
                        remapped = np.where(vals > d_bin, vals - 1, vals)
                        col_out[mask] = (remapped + off).astype(np.uint8)
                X_out[:, b_idx] = col_out
        return X_out

    def fit_transform(self, X_binned: np.ndarray, n_bins_per_feature: np.ndarray) -> np.ndarray:
        return self.fit(X_binned, n_bins_per_feature).transform(X_binned)

print("[OK] Đã định nghĩa HistogramBinner & ExclusiveFeatureBundler (EFB).")

In [ ]:
class _TreeNode:
    """Cấu trúc nút cây tối ưu bộ nhớ bằng __slots__."""

    __slots__ = (
        "node_id", "depth", "is_leaf", "leaf_value",
        "feature", "bin_thresh", "gain", "left", "right",
        "sample_idx", "G_hist", "H_hist", "C_hist",
        "G_sum", "H_sum", "n_samples",
    )

    def __init__(self, node_id: int, depth: int, leaf_value: float, sample_idx=None):
        self.node_id = node_id
        self.depth = depth
        self.is_leaf = True
        self.leaf_value = leaf_value
        self.feature = -1
        self.bin_thresh = -1
        self.gain = 0.0
        self.left = None
        self.right = None
        self.sample_idx = sample_idx
        self.G_hist = None
        self.H_hist = None
        self.C_hist = None
        self.G_sum = 0.0
        self.H_sum = 0.0
        self.n_samples = 0


class LeafWiseHistogramTree:
    """Bước 3 của LightGBM: Cây quyết định Histogram + Leaf-Wise Growth + Histogram Subtraction."""

    def __init__(
        self,
        num_leaves: int = 31,
        max_depth: int = -1,
        min_child_samples: int = 50,
        min_child_weight: float = 1e-3,
        reg_alpha: float = 0.1,
        reg_lambda: float = 5.0,
        min_split_gain: float = 0.0,
        colsample_bytree: float = 0.8,
        use_hist_subtraction: bool = True,
        growth_mode: str = "leaf_wise",  # "leaf_wise" (LightGBM) hoặc "level_wise" (đối chiếu)
    ):
        self.num_leaves = num_leaves
        self.max_depth = max_depth
        self.min_child_samples = min_child_samples
        self.min_child_weight = min_child_weight
        self.reg_alpha = reg_alpha
        self.reg_lambda = reg_lambda
        self.min_split_gain = min_split_gain
        self.colsample_bytree = colsample_bytree
        self.use_hist_subtraction = use_hist_subtraction
        self.growth_mode = growth_mode

        self.root_ = None
        self.n_leaves_ = 0
        self.split_importances_ = None
        self.gain_importances_ = None

    @staticmethod
    def _soft_threshold(g, alpha: float):
        """Toán tử ngưỡng mềm cho điều chuẩn L1 (reg_alpha)."""
        if alpha <= 0.0:
            return g
        return np.sign(g) * np.maximum(np.abs(g) - alpha, 0.0)

    def _calc_leaf_value(self, G: float, H: float) -> float:
        """Trọng số lá tối ưu theo công thức Newton bậc 2 có L1/L2 regularization."""
        g_adj = float(self._soft_threshold(G, self.reg_alpha))
        return -g_adj / (H + self.reg_lambda)

    def _build_histograms(
        self,
        X_T: np.ndarray,
        feat_indices: np.ndarray,
        sample_idx: np.ndarray,
        g: np.ndarray,
        h: np.ndarray,
        max_b: int,
    ):
        """Dựng histogram (G, H, Count) bằng np.bincount trên bộ nhớ liên tục theo từng đặc trưng."""
        n_f = len(feat_indices)
        G_hist = np.zeros((n_f, max_b), dtype=np.float64)
        H_hist = np.zeros((n_f, max_b), dtype=np.float64)
        C_hist = np.zeros((n_f, max_b), dtype=np.int32)

        g_sub = g[sample_idx]
        h_sub = h[sample_idx]

        for k, f in enumerate(feat_indices):
            bins = X_T[f, sample_idx]
            G_hist[k] = np.bincount(bins, weights=g_sub, minlength=max_b)
            H_hist[k] = np.bincount(bins, weights=h_sub, minlength=max_b)
            C_hist[k] = np.bincount(bins, minlength=max_b)

        return G_hist, H_hist, C_hist

    def _find_best_split_from_hist(
        self,
        G_hist: np.ndarray,
        H_hist: np.ndarray,
        C_hist: np.ndarray,
        G_total: float,
        H_total: float,
        n_total: int,
        feat_indices: np.ndarray,
        bundle_n_bins: np.ndarray,
    ):
        """Quét tìm (feature, bin_thresh) có Gain lớn nhất theo toàn bộ ma trận (n_f, max_b - 1)."""
        G_L = np.cumsum(G_hist, axis=1)[:, :-1]
        H_L = np.cumsum(H_hist, axis=1)[:, :-1]
        C_L = np.cumsum(C_hist, axis=1)[:, :-1]

        G_R = G_total - G_L
        H_R = H_total - H_L
        C_R = n_total - C_L

        max_b_minus_1 = G_L.shape[1]
        bin_idx_grid = np.arange(max_b_minus_1)[None, :]
        valid_bins = bin_idx_grid < (bundle_n_bins[feat_indices, None] - 1)

        valid = (
            valid_bins
            & (C_L >= self.min_child_samples)
            & (C_R >= self.min_child_samples)
            & (H_L >= self.min_child_weight)
            & (H_R >= self.min_child_weight)
        )
        if not np.any(valid):
            return -1, -1, 0.0

        lam = self.reg_lambda
        alpha = self.reg_alpha
        parent_score = (self._soft_threshold(G_total, alpha) ** 2) / (H_total + lam)
        left_score = (self._soft_threshold(G_L, alpha) ** 2) / (H_L + lam)
        right_score = (self._soft_threshold(G_R, alpha) ** 2) / (H_R + lam)

        gains = 0.5 * (left_score + right_score - parent_score) - self.min_split_gain
        gains = np.where(valid, gains, -np.inf)

        flat_idx = int(np.argmax(gains))
        best_gain = float(gains.ravel()[flat_idx])
        if best_gain <= 0.0:
            return -1, -1, 0.0

        k_best, b_best = divmod(flat_idx, max_b_minus_1)
        return int(feat_indices[k_best]), int(b_best), best_gain

    def fit(
        self,
        X_T: np.ndarray,
        bundle_n_bins: np.ndarray,
        sample_idx: np.ndarray,
        g: np.ndarray,
        h: np.ndarray,
        bin_to_orig_feature: list,
        n_orig_features: int,
        rng: np.random.Generator,
    ):
        n_bundled = X_T.shape[0]
        self.split_importances_ = np.zeros(n_orig_features, dtype=np.float64)
        self.gain_importances_ = np.zeros(n_orig_features, dtype=np.float64)

        # Feature Fraction (colsample_bytree)
        if self.colsample_bytree < 1.0:
            n_sub = max(1, int(np.round(n_bundled * self.colsample_bytree)))
            feat_indices = np.sort(rng.choice(n_bundled, size=n_sub, replace=False))
        else:
            feat_indices = np.arange(n_bundled, dtype=np.int32)

        max_b = int(bundle_n_bins[feat_indices].max())

        # Nút gốc
        G_root = float(g[sample_idx].sum())
        H_root = float(h[sample_idx].sum())
        n_root = len(sample_idx)

        root = _TreeNode(0, 0, self._calc_leaf_value(G_root, H_root), sample_idx)
        root.G_sum, root.H_sum, root.n_samples = G_root, H_root, n_root
        root.G_hist, root.H_hist, root.C_hist = self._build_histograms(
            X_T, feat_indices, sample_idx, g, h, max_b
        )
        self.root_ = root
        self.n_leaves_ = 1
        next_node_id = 1

        if (self.max_depth < 0 or 0 < self.max_depth) and n_root >= 2 * self.min_child_samples:
            f, b, gain = self._find_best_split_from_hist(
                root.G_hist, root.H_hist, root.C_hist,
                G_root, H_root, n_root, feat_indices, bundle_n_bins
            )
        else:
            f, b, gain = -1, -1, 0.0

        # Max-Heap ưu tiên lá có Gain lớn nhất (Leaf-wise) hoặc ưu tiên theo tầng (Level-wise)
        heap = []
        if f != -1:
            prio = (-gain, root.node_id) if self.growth_mode == "leaf_wise" else (root.depth, -gain, root.node_id)
            heapq.heappush(heap, (prio, f, b, gain, root))

        while heap and self.n_leaves_ < self.num_leaves:
            _, best_f, best_b, best_gain, node = heapq.heappop(heap)

            idx = node.sample_idx
            left_mask = X_T[best_f, idx] <= best_b
            left_idx = idx[left_mask]
            right_idx = idx[~left_mask]

            if len(left_idx) < self.min_child_samples or len(right_idx) < self.min_child_samples:
                continue

            node.is_leaf = False
            node.feature = best_f
            node.bin_thresh = best_b
            node.gain = best_gain

            # Quy chiếu tầm quan trọng về đặc trưng gốc ban đầu (EFB-aware)
            orig_f = int(bin_to_orig_feature[best_f][best_b])
            self.split_importances_[orig_f] += 1.0
            self.gain_importances_[orig_f] += best_gain

            G_left = float(g[left_idx].sum())
            H_left = float(h[left_idx].sum())
            G_right = node.G_sum - G_left
            H_right = node.H_sum - H_left

            left_child = _TreeNode(next_node_id, node.depth + 1, self._calc_leaf_value(G_left, H_left), left_idx)
            left_child.G_sum, left_child.H_sum, left_child.n_samples = G_left, H_left, len(left_idx)
            next_node_id += 1

            right_child = _TreeNode(next_node_id, node.depth + 1, self._calc_leaf_value(G_right, H_right), right_idx)
            right_child.G_sum, right_child.H_sum, right_child.n_samples = G_right, H_right, len(right_idx)
            next_node_id += 1

            node.left = left_child
            node.right = right_child
            self.n_leaves_ += 1

            can_split_further = (self.n_leaves_ < self.num_leaves) and (
                self.max_depth < 0 or left_child.depth < self.max_depth
            )

            if can_split_further:
                if self.use_hist_subtraction:
                    # HISTOGRAM SUBTRACTION TRICK: chỉ dựng histogram cho nút con ít mẫu hơn!
                    if left_child.n_samples <= right_child.n_samples:
                        left_child.G_hist, left_child.H_hist, left_child.C_hist = self._build_histograms(
                            X_T, feat_indices, left_idx, g, h, max_b
                        )
                        right_child.G_hist = node.G_hist - left_child.G_hist
                        right_child.H_hist = node.H_hist - left_child.H_hist
                        right_child.C_hist = node.C_hist - left_child.C_hist
                    else:
                        right_child.G_hist, right_child.H_hist, right_child.C_hist = self._build_histograms(
                            X_T, feat_indices, right_idx, g, h, max_b
                        )
                        left_child.G_hist = node.G_hist - right_child.G_hist
                        left_child.H_hist = node.H_hist - right_child.H_hist
                        left_child.C_hist = node.C_hist - right_child.C_hist
                else:
                    left_child.G_hist, left_child.H_hist, left_child.C_hist = self._build_histograms(
                        X_T, feat_indices, left_idx, g, h, max_b
                    )
                    right_child.G_hist, right_child.H_hist, right_child.C_hist = self._build_histograms(
                        X_T, feat_indices, right_idx, g, h, max_b
                    )

                for child in (left_child, right_child):
                    if child.n_samples >= 2 * self.min_child_samples:
                        cf, cb, cgain = self._find_best_split_from_hist(
                            child.G_hist, child.H_hist, child.C_hist,
                            child.G_sum, child.H_sum, child.n_samples,
                            feat_indices, bundle_n_bins
                        )
                        if cf != -1:
                            prio = (-cgain, child.node_id) if self.growth_mode == "leaf_wise" else (child.depth, -cgain, child.node_id)
                            heapq.heappush(heap, (prio, cf, cb, cgain, child))

            # Giải phóng histogram của nút cha ngay sau khi tách
            node.sample_idx = None
            node.G_hist = node.H_hist = node.C_hist = None

        self._cleanup_node(self.root_)
        return self

    def _cleanup_node(self, node):
        if node is None:
            return
        node.sample_idx = None
        node.G_hist = node.H_hist = node.C_hist = None
        if not node.is_leaf:
            self._cleanup_node(node.left)
            self._cleanup_node(node.right)

    def predict_binned(self, X_bundled: np.ndarray) -> np.ndarray:
        out = np.empty(X_bundled.shape[0], dtype=np.float64)
        self._predict_recursive(self.root_, X_bundled, np.arange(X_bundled.shape[0]), out)
        return out

    def _predict_recursive(self, node, X_bundled, idx, out):
        if len(idx) == 0:
            return
        if node.is_leaf:
            out[idx] = node.leaf_value
            return
        left_mask = X_bundled[idx, node.feature] <= node.bin_thresh
        self._predict_recursive(node.left, X_bundled, idx[left_mask], out)
        self._predict_recursive(node.right, X_bundled, idx[~left_mask], out)

print("[OK] Đã định nghĩa LeafWiseHistogramTree.")

In [ ]:
class LightGBMScratch:
    """Thuật toán LightGBM hoàn chỉnh tự code bằng Pure NumPy (Histogram + Leaf-wise + GOSS + EFB)."""

    def __init__(
        self,
        n_estimators: int = 150,
        learning_rate: float = 0.06,
        num_leaves: int = 31,
        max_depth: int = -1,
        min_child_samples: int = 50,
        min_child_weight: float = 1e-3,
        max_bins: int = 63,
        reg_alpha: float = 0.1,
        reg_lambda: float = 5.0,
        min_split_gain: float = 0.0,
        colsample_bytree: float = 0.8,
        scale_pos_weight: float = 1.0,
        use_goss: bool = True,
        top_rate: float = 0.2,
        other_rate: float = 0.1,
        goss_warmup_rounds: int = 10,
        use_efb: bool = True,
        efb_sparse_threshold: float = 0.85,
        efb_max_conflict_rate: float = 0.001,
        use_hist_subtraction: bool = True,
        growth_mode: str = "leaf_wise",
        random_state: int = 1,
    ):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.num_leaves = num_leaves
        self.max_depth = max_depth
        self.min_child_samples = min_child_samples
        self.min_child_weight = min_child_weight
        self.max_bins = max_bins
        self.reg_alpha = reg_alpha
        self.reg_lambda = reg_lambda
        self.min_split_gain = min_split_gain
        self.colsample_bytree = colsample_bytree
        self.scale_pos_weight = scale_pos_weight
        self.use_goss = use_goss
        self.top_rate = top_rate
        self.other_rate = other_rate
        self.goss_warmup_rounds = goss_warmup_rounds
        self.use_efb = use_efb
        self.efb_sparse_threshold = efb_sparse_threshold
        self.efb_max_conflict_rate = efb_max_conflict_rate
        self.use_hist_subtraction = use_hist_subtraction
        self.growth_mode = growth_mode
        self.random_state = random_state

        self.binner_ = None
        self.bundler_ = None
        self.init_score_ = 0.0
        self.trees_ = []
        self.best_iteration_ = 0
        self.feature_importances_ = None
        self.split_importances_ = None
        self.evals_result_ = {}

    @staticmethod
    def _sigmoid(x):
        return 1.0 / (1.0 + np.exp(-np.clip(x, -30.0, 30.0)))

    @staticmethod
    def _weighted_logloss(y, p, sw):
        p_clip = np.clip(p, 1e-12, 1.0 - 1e-12)
        loss = -(y * np.log(p_clip) + (1.0 - y) * np.log(1.0 - p_clip))
        return float(np.sum(sw * loss) / np.sum(sw))

    def _goss_sample(self, g: np.ndarray, h: np.ndarray, rng: np.random.Generator):
        """GOSS — Gradient-based One-Side Sampling (Algorithm 2, Ke et al., 2017)."""
        n = len(g)
        top_n = max(1, int(self.top_rate * n))
        other_n = max(1, int(self.other_rate * n))
        if top_n + other_n >= n:
            return np.arange(n, dtype=np.int32), g, h

        abs_g = np.abs(g)
        # Phân hoạch tuyến tính O(N) lấy top_n mẫu có |g_i| lớn nhất
        part = np.argpartition(abs_g, n - top_n)
        top_idx = part[n - top_n:]
        rest_idx = part[:n - top_n]

        # Lấy mẫu ngẫu nhiên other_n mẫu từ nhóm gradient nhỏ
        rand_idx = rng.choice(rest_idx, size=other_n, replace=False)

        # Nhân hệ số bù (1 - a) / b lên nhóm gradient nhỏ để bảo toàn ước lượng Gain không chệch
        fact = (1.0 - self.top_rate) / self.other_rate
        g_mod = g.copy()
        h_mod = h.copy()
        g_mod[rand_idx] *= fact
        h_mod[rand_idx] *= fact

        used_idx = np.concatenate([top_idx, rand_idx]).astype(np.int32)
        return used_idx, g_mod, h_mod

    def fit(self, X, y, eval_set=None, early_stopping_rounds=None, verbose=False):
        rng = np.random.default_rng(self.random_state)
        X = np.asarray(X, dtype=np.float32)
        y = np.asarray(y, dtype=np.float64)
        n_samples, n_features = X.shape

        # Bước 1: Histogram Binning + EFB
        self.binner_ = HistogramBinner(max_bins=self.max_bins, random_state=self.random_state)
        X_binned = self.binner_.fit_transform(X)

        self.bundler_ = ExclusiveFeatureBundler(
            sparse_threshold=self.efb_sparse_threshold,
            max_conflict_rate=self.efb_max_conflict_rate,
            enabled=self.use_efb,
        )
        X_bundled = self.bundler_.fit_transform(X_binned, self.binner_.n_bins_per_feature_)
        X_T = np.ascontiguousarray(X_bundled.T)

        has_val = eval_set is not None
        if has_val:
            X_val, y_val = eval_set
            X_val_bundled = self.bundler_.transform(
                self.binner_.transform(np.asarray(X_val, dtype=np.float32))
            )
            y_val = np.asarray(y_val, dtype=np.float64)
            sw_val = np.where(y_val == 1.0, self.scale_pos_weight, 1.0)

        # Bước 2: Khởi tạo log-odds F_0 có xét trọng số lớp
        sw = np.where(y == 1.0, self.scale_pos_weight, 1.0)
        p0 = float(np.sum(sw * y) / np.sum(sw))
        p0 = np.clip(p0, 1e-6, 1.0 - 1e-6)
        self.init_score_ = float(np.log(p0 / (1.0 - p0)))

        raw_preds = np.full(n_samples, self.init_score_, dtype=np.float64)
        if has_val:
            val_raw_preds = np.full(len(y_val), self.init_score_, dtype=np.float64)

        self.trees_ = []
        self.evals_result_ = {"train_logloss": [], "val_logloss": [], "val_auc": []}

        best_val_auc = -np.inf
        best_iter = 0
        no_improve = 0
        all_idx = np.arange(n_samples, dtype=np.int32)

        # Bước 3: Vòng lặp Gradient Boosting
        for m in range(self.n_estimators):
            probs = self._sigmoid(raw_preds)
            g = sw * (probs - y)
            h = np.maximum(sw * probs * (1.0 - probs), 1e-12)

            if self.use_goss and m >= self.goss_warmup_rounds:
                used_idx, g_use, h_use = self._goss_sample(g, h, rng)
            else:
                used_idx, g_use, h_use = all_idx, g, h

            tree = LeafWiseHistogramTree(
                num_leaves=self.num_leaves,
                max_depth=self.max_depth,
                min_child_samples=self.min_child_samples,
                min_child_weight=self.min_child_weight,
                reg_alpha=self.reg_alpha,
                reg_lambda=self.reg_lambda,
                min_split_gain=self.min_split_gain,
                colsample_bytree=self.colsample_bytree,
                use_hist_subtraction=self.use_hist_subtraction,
                growth_mode=self.growth_mode,
            ).fit(
                X_T=X_T,
                bundle_n_bins=self.bundler_.bundle_n_bins_,
                sample_idx=used_idx,
                g=g_use,
                h=h_use,
                bin_to_orig_feature=self.bundler_.bin_to_orig_feature_,
                n_orig_features=n_features,
                rng=rng,
            )

            self.trees_.append(tree)
            raw_preds += self.learning_rate * tree.predict_binned(X_bundled)
            tr_loss = self._weighted_logloss(y, self._sigmoid(raw_preds), sw)
            self.evals_result_["train_logloss"].append(tr_loss)

            if has_val:
                val_raw_preds += self.learning_rate * tree.predict_binned(X_val_bundled)
                val_probs = self._sigmoid(val_raw_preds)
                val_loss = self._weighted_logloss(y_val, val_probs, sw_val)
                val_auc = float(roc_auc_score(y_val, val_probs))
                self.evals_result_["val_logloss"].append(val_loss)
                self.evals_result_["val_auc"].append(val_auc)

                if val_auc > best_val_auc + 1e-5:
                    best_val_auc = val_auc
                    best_iter = m + 1
                    no_improve = 0
                else:
                    no_improve += 1

                if verbose and ((m + 1) % 20 == 0 or m == 0):
                    print(f"[Iter {m+1:3d}] train_loss={tr_loss:.4f} | val_loss={val_loss:.4f} | val_auc={val_auc:.4f}")

                if early_stopping_rounds is not None and no_improve >= early_stopping_rounds:
                    if verbose:
                        print(f"--> Early stopping tại vòng {m+1} (best_iter={best_iter}, best_val_auc={best_val_auc:.4f})")
                    break
            else:
                best_iter = m + 1
                if verbose and ((m + 1) % 20 == 0 or m == 0):
                    print(f"[Iter {m+1:3d}] train_loss={tr_loss:.4f}")

        self.best_iteration_ = best_iter
        total_gain_imp = np.sum([t.gain_importances_ for t in self.trees_[:best_iter]], axis=0)
        total_split_imp = np.sum([t.split_importances_ for t in self.trees_[:best_iter]], axis=0)

        self.split_importances_ = total_split_imp
        self.feature_importances_ = (
            total_gain_imp / total_gain_imp.sum() if total_gain_imp.sum() > 0 else total_gain_imp
        )
        return self

    def predict_proba(self, X, num_iteration=None):
        X_bundled = self.bundler_.transform(
            self.binner_.transform(np.asarray(X, dtype=np.float32))
        )
        n_trees = self.best_iteration_ if num_iteration is None else min(num_iteration, len(self.trees_))
        raw = np.full(X_bundled.shape[0], self.init_score_, dtype=np.float64)
        for tree in self.trees_[:n_trees]:
            raw += self.learning_rate * tree.predict_binned(X_bundled)
        p1 = self._sigmoid(raw)
        return np.column_stack([1.0 - p1, p1])

    def predict(self, X, threshold: float = 0.5):
        return (self.predict_proba(X)[:, 1] >= threshold).astype(int)

print("[OK] Đã định nghĩa hoàn chỉnh LightGBMScratch (Pure NumPy).")

## 3. Kiểm chứng thuật toán & Ablation Study (trên tập Validation nội bộ)

Trước khi huấn luyện mô hình chính thức, ta kiểm chứng tính đúng đắn toán học và hiệu quả của từng kỹ thuật cốt lõi trong `LightGBMScratch`:
1. **Histogram Subtraction Trick:** Kiểm chứng độ sai lệch xác suất giữa có/không dùng Histogram Subtraction (phải $= 0$ tuyệt đối) và đo thời gian tăng tốc.
2. **Leaf-Wise vs Level-Wise:** So sánh chất lượng cây cùng `num_leaves=31`.
3. **GOSS vs Full Sampling:** So sánh tốc độ huấn luyện và độ giữ vững Val ROC-AUC.
4. **EFB Compression:** Kiểm tra số lượng đặc trưng thưa được gộp thành bundle trên bộ đặc trưng Home Credit.

In [ ]:
ablation_rows = []
N_ABLA_TREES = 30

# 1. Không dùng Histogram Subtraction (Full sampling, Leaf-wise)
t0 = time.perf_counter()
m_nosub = LightGBMScratch(
    n_estimators=N_ABLA_TREES, learning_rate=0.08, num_leaves=31,
    scale_pos_weight=SCALE_POS_WEIGHT, use_goss=False, use_efb=False,
    use_hist_subtraction=False, growth_mode="leaf_wise", random_state=RANDOM_STATE,
).fit(X_tr, y_tr)
t_nosub = time.perf_counter() - t0
p_nosub = m_nosub.predict_proba(X_val)[:, 1]
ablation_rows.append({
    "Cấu hình": "1. Leaf-wise (No Subtraction, No GOSS, No EFB)",
    "Val ROC-AUC": roc_auc_score(y_val, p_nosub),
    "Val BalAcc (t=0.5)": balanced_accuracy_score(y_val, (p_nosub >= 0.5).astype(int)),
    "Train Time (s)": t_nosub,
    "Features Used": m_nosub.bundler_.n_bundled_features_,
})

# 2. Bật Histogram Subtraction Trick (kiểm chứng sai số số học == 0)
t0 = time.perf_counter()
m_sub = LightGBMScratch(
    n_estimators=N_ABLA_TREES, learning_rate=0.08, num_leaves=31,
    scale_pos_weight=SCALE_POS_WEIGHT, use_goss=False, use_efb=False,
    use_hist_subtraction=True, growth_mode="leaf_wise", random_state=RANDOM_STATE,
).fit(X_tr, y_tr)
t_sub = time.perf_counter() - t0
p_sub = m_sub.predict_proba(X_val)[:, 1]
max_diff = float(np.max(np.abs(p_sub - p_nosub)))
print(f"[Kiểm chứng Histogram Subtraction] Max |p_sub - p_nosub| = {max_diff:.2e} | Tăng tốc: {t_nosub / max(t_sub, 1e-6):.2f}x")
assert max_diff < 1e-9, "Histogram Subtraction phải cho kết quả giống hệt về mặt toán học!"
ablation_rows.append({
    "Cấu hình": "2. + Histogram Subtraction Trick",
    "Val ROC-AUC": roc_auc_score(y_val, p_sub),
    "Val BalAcc (t=0.5)": balanced_accuracy_score(y_val, (p_sub >= 0.5).astype(int)),
    "Train Time (s)": t_sub,
    "Features Used": m_sub.bundler_.n_bundled_features_,
})

# 3. Đối chiếu Level-wise (cùng num_leaves=31)
t0 = time.perf_counter()
m_lvl = LightGBMScratch(
    n_estimators=N_ABLA_TREES, learning_rate=0.08, num_leaves=31,
    scale_pos_weight=SCALE_POS_WEIGHT, use_goss=False, use_efb=False,
    use_hist_subtraction=True, growth_mode="level_wise", random_state=RANDOM_STATE,
).fit(X_tr, y_tr)
t_lvl = time.perf_counter() - t0
p_lvl = m_lvl.predict_proba(X_val)[:, 1]
ablation_rows.append({
    "Cấu hình": "3. Level-wise (cùng 31 lá, để đối chiếu)",
    "Val ROC-AUC": roc_auc_score(y_val, p_lvl),
    "Val BalAcc (t=0.5)": balanced_accuracy_score(y_val, (p_lvl >= 0.5).astype(int)),
    "Train Time (s)": t_lvl,
    "Features Used": m_lvl.bundler_.n_bundled_features_,
})

# 4. Full LightGBMScratch (+ Histogram Subtraction + GOSS + EFB)
t0 = time.perf_counter()
m_full = LightGBMScratch(
    n_estimators=N_ABLA_TREES, learning_rate=0.08, num_leaves=31,
    scale_pos_weight=SCALE_POS_WEIGHT, use_goss=True, top_rate=0.2, other_rate=0.1,
    use_efb=True, use_hist_subtraction=True, growth_mode="leaf_wise", random_state=RANDOM_STATE,
).fit(X_tr, y_tr)
t_full = time.perf_counter() - t0
p_full = m_full.predict_proba(X_val)[:, 1]
ablation_rows.append({
    "Cấu hình": "4. Full LightGBMScratch (+ Subtraction + GOSS + EFB)",
    "Val ROC-AUC": roc_auc_score(y_val, p_full),
    "Val BalAcc (t=0.5)": balanced_accuracy_score(y_val, (p_full >= 0.5).astype(int)),
    "Train Time (s)": t_full,
    "Features Used": m_full.bundler_.n_bundled_features_,
})

ablation_df = pd.DataFrame(ablation_rows)
display(ablation_df.round(4))
print(
    f"[EFB] Nén từ {m_full.bundler_.n_orig_features_} -> {m_full.bundler_.n_bundled_features_} đặc trưng "
    f"| Tổng tăng tốc so với bản cơ bản: {t_nosub / max(t_full, 1e-6):.2f}x"
)

## 4. Huấn luyện mô hình chính thức: `LightGBMScratch` (Pure NumPy)

- Huấn luyện tối đa **180 cây** (`learning_rate=0.05`, `num_leaves=31`, `max_bins=63`, `min_child_samples=60`, `reg_alpha=0.2`, `reg_lambda=5.0`, `colsample_bytree=0.8`, `scale_pos_weight=neg/pos`) tích hợp đầy đủ **Histogram Subtraction + GOSS + EFB**.
- Áp dụng **Early Stopping (`patience=25`)** theo **Validation ROC-AUC** trên tập `X_val`.
- Quét tìm ngưỡng quyết định tối ưu $t^*$ theo **Balanced Accuracy** trên tập `X_val` (tuyệt đối không chạm vào `X_test`).

In [ ]:
t0 = time.perf_counter()
lgbm_scratch = LightGBMScratch(
    n_estimators=180,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=8,
    min_child_samples=60,
    max_bins=63,
    reg_alpha=0.2,
    reg_lambda=5.0,
    colsample_bytree=0.8,
    scale_pos_weight=SCALE_POS_WEIGHT,
    use_goss=True,
    top_rate=0.2,
    other_rate=0.1,
    goss_warmup_rounds=15,
    use_efb=True,
    use_hist_subtraction=True,
    growth_mode="leaf_wise",
    random_state=RANDOM_STATE,
).fit(
    X_tr, y_tr,
    eval_set=(X_val, y_val),
    early_stopping_rounds=25,
    verbose=True,
)
t_scratch = time.perf_counter() - t0

# Dự đoán xác suất trên Validation và Test
val_prob_scratch = lgbm_scratch.predict_proba(X_val)[:, 1]
y_prob_scratch = lgbm_scratch.predict_proba(X_test)[:, 1]

# Quét tìm ngưỡng tối ưu Balanced Accuracy duy nhất trên tập Validation (không leak Test)
thresholds = np.arange(0.20, 0.805, 0.005)
val_bal_accs = [balanced_accuracy_score(y_val, (val_prob_scratch >= t).astype(int)) for t in thresholds]
best_idx = int(np.argmax(val_bal_accs))
best_thresh_scratch = float(thresholds[best_idx])

print(f"\n[Done] LightGBMScratch — Thời gian: {t_scratch:.2f}s | Best iteration: {lgbm_scratch.best_iteration_}")
print(f"  -> Val ROC-AUC           : {roc_auc_score(y_val, val_prob_scratch):.4f}")
print(f"  -> Val BalAcc (t=0.500)  : {balanced_accuracy_score(y_val, (val_prob_scratch >= 0.5).astype(int)):.4f}")
print(f"  -> Val BalAcc (t*={best_thresh_scratch:.3f}): {val_bal_accs[best_idx]:.4f}")

# Áp dụng ngưỡng t* tìm trên Val xuống Test
y_pred_scratch = (y_prob_scratch >= best_thresh_scratch).astype(int)

# Vẽ Learning Curves (Log-Loss & Validation ROC-AUC)
iters = np.arange(1, len(lgbm_scratch.evals_result_["train_logloss"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.4))

axes[0].plot(iters, lgbm_scratch.evals_result_["train_logloss"], label="Train Weighted Log-Loss", lw=2)
axes[0].plot(iters, lgbm_scratch.evals_result_["val_logloss"], label="Val Weighted Log-Loss", lw=2)
axes[0].axvline(lgbm_scratch.best_iteration_, color="red", linestyle="--", label=f"Best Iter ({lgbm_scratch.best_iteration_})")
axes[0].set(xlabel="Boosting Iteration", ylabel="Weighted Log-Loss", title="Learning Curve — Weighted Log-Loss")
axes[0].legend()

axes[1].plot(iters, lgbm_scratch.evals_result_["val_auc"], color="darkorange", lw=2, label="Val ROC-AUC")
axes[1].axvline(lgbm_scratch.best_iteration_, color="red", linestyle="--", label=f"Best Iter ({lgbm_scratch.best_iteration_})")
axes[1].set(xlabel="Boosting Iteration", ylabel="ROC-AUC", title="Learning Curve — Validation ROC-AUC")
axes[1].legend()

plt.tight_layout()
plt.show()

## 5. Đối chiếu công bằng: `LightGBMScratch` (Pure NumPy) vs `lightgbm.LGBMClassifier` (Library)

Huấn luyện thư viện `lightgbm.LGBMClassifier` chính thức với **cùng siêu tham số, cùng tập `(X_tr, y_tr, X_val, y_val)` và cùng cơ chế chọn ngưỡng trên Validation** để kiểm chứng mức độ tương đồng của bản tự cài đặt bằng tay.

In [ ]:
has_lgbm_lib = False
y_prob_lib = None
y_pred_lib = None

try:
    import lightgbm as lgb
    has_lgbm_lib = True

    t0 = time.perf_counter()
    lgbm_lib = lgb.LGBMClassifier(
        n_estimators=180,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=8,
        min_child_samples=60,
        max_bin=63,
        reg_alpha=0.2,
        reg_lambda=5.0,
        colsample_bytree=0.8,
        scale_pos_weight=SCALE_POS_WEIGHT,
        metric="auc",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=-1,
    )
    lgbm_lib.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        eval_metric="auc",
        callbacks=[lgb.early_stopping(25, first_metric_only=True, verbose=False)],
    )
    t_lib = time.perf_counter() - t0

    val_prob_lib = lgbm_lib.predict_proba(X_val)[:, 1]
    val_bal_lib = [balanced_accuracy_score(y_val, (val_prob_lib >= t).astype(int)) for t in thresholds]
    best_thresh_lib = float(thresholds[int(np.argmax(val_bal_lib))])

    y_prob_lib = lgbm_lib.predict_proba(X_test)[:, 1]
    y_pred_lib = (y_prob_lib >= best_thresh_lib).astype(int)
    print(f"[Done] LGBMClassifier (Library) — Thời gian: {t_lib:.2f}s | Best iter: {lgbm_lib.best_iteration_} | t*={best_thresh_lib:.3f}")
except ImportError:
    print("[Info] Thư viện `lightgbm` chưa được cài trong môi trường này — bỏ qua đối chiếu Library (chỉ chạy bản From Scratch).")

def summarize_model(name, source, y_true, y_pred, y_prob, train_time, thresh):
    return {
        "Model": name,
        "Source": source,
        "Threshold (Val t*)": round(thresh, 3),
        "Balanced Accuracy": balanced_accuracy_score(y_true, y_pred),
        "ROC-AUC": roc_auc_score(y_true, y_prob),
        "PR-AUC (AP)": average_precision_score(y_true, y_prob),
        "High-Risk Recall": recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        "High-Risk Precision": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        "High-Risk F1": f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        "Overall Accuracy": accuracy_score(y_true, y_pred),
        "Train Time (s)": train_time,
    }

compare_rows = [
    summarize_model(
        "LightGBM (From Scratch — Pure NumPy)", "From Scratch",
        y_test, y_pred_scratch, y_prob_scratch, t_scratch, best_thresh_scratch,
    )
]
if has_lgbm_lib:
    compare_rows.append(
        summarize_model(
            "LGBMClassifier (Library)", "Library",
            y_test, y_pred_lib, y_prob_lib, t_lib, best_thresh_lib,
        )
    )

compare_df = pd.DataFrame(compare_rows)
display(compare_df.round(4))

## 6. Đánh giá chính thức bằng `src/metrics.py` & Lưu kết quả JSON của Member 3

- Chấm điểm mô hình **`LightGBM (From Scratch)`** bằng hàm dùng chung `evaluate_and_save()` → xuất file chuẩn `results/member3_lightgbm.json` phục vụ tổng hợp qua `compare_models.py`.
- Trực quan hóa **Confusion Matrix**, **ROC Curve** và **Precision-Recall Curve** trên tập Hold-out Test.

In [ ]:
# Đánh giá chính thức và lưu kết quả của Thành viên 3 vào results/member3_lightgbm.json
metrics_member3 = evaluate_and_save(
    model_name="LightGBM (From Scratch)",
    filename="member3_lightgbm",
    y_true=y_test,
    y_pred=y_pred_scratch,
    y_prob=y_prob_scratch,
)

# Vẽ Confusion Matrix theo chuẩn dùng chung
plot_confusion_matrix(y_test, y_pred_scratch, model_name="LightGBM (From Scratch)")

# Vẽ ROC Curve & Precision-Recall Curve
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6))

fpr_s, tpr_s, _ = roc_curve(y_test, y_prob_scratch)
axes[0].plot(fpr_s, tpr_s, lw=2.2, label=f"LightGBM Scratch (AUC={roc_auc_score(y_test, y_prob_scratch):.4f})")
if has_lgbm_lib:
    fpr_l, tpr_l, _ = roc_curve(y_test, y_prob_lib)
    axes[0].plot(fpr_l, tpr_l, lw=1.8, linestyle="--", label=f"LGBM Library (AUC={roc_auc_score(y_test, y_prob_lib):.4f})")
axes[0].plot([0, 1], [0, 1], ":", color="gray", label="Random")
axes[0].set(xlabel="False Positive Rate", ylabel="True Positive Rate", title="ROC Curve (Hold-out Test)")
axes[0].legend(loc="lower right")

prec_s, rec_s, _ = precision_recall_curve(y_test, y_prob_scratch)
axes[1].plot(rec_s, prec_s, lw=2.2, label=f"LightGBM Scratch (AP={average_precision_score(y_test, y_prob_scratch):.4f})")
if has_lgbm_lib:
    prec_l, rec_l, _ = precision_recall_curve(y_test, y_prob_lib)
    axes[1].plot(rec_l, prec_l, lw=1.8, linestyle="--", label=f"LGBM Library (AP={average_precision_score(y_test, y_prob_lib):.4f})")
axes[1].axhline(y_test.mean(), color="gray", linestyle=":", label=f"Baseline ({y_test.mean():.3f})")
axes[1].set(xlabel="Recall (High-Risk)", ylabel="Precision (High-Risk)", title="Precision-Recall Curve (Hold-out Test)")
axes[1].legend(loc="upper right")

plt.tight_layout()
plt.show()

In [ ]:
# Top 20 đặc trưng quan trọng nhất (Gain & Split Count) từ LightGBMScratch
fi_df = pd.DataFrame({
    "Feature": feature_names,
    "Gain Importance": lgbm_scratch.feature_importances_,
    "Split Count": lgbm_scratch.split_importances_.astype(int),
}).sort_values("Gain Importance", ascending=False).reset_index(drop=True)

display(fi_df.head(20))

fig, axes = plt.subplots(1, 2, figsize=(14, 6.2))
top_gain = fi_df.head(20).iloc[::-1]
axes[0].barh(top_gain["Feature"], top_gain["Gain Importance"], color="#1f77b4")
axes[0].set(xlabel="Normalized Total Gain", title="Top 20 Feature Importance — Gain (LightGBM Scratch)")

top_split = fi_df.sort_values("Split Count", ascending=False).head(20).iloc[::-1]
axes[1].barh(top_split["Feature"], top_split["Split Count"], color="#ff7f0e")
axes[1].set(xlabel="Number of Splits", title="Top 20 Feature Importance — Split Count (LightGBM Scratch)")

plt.tight_layout()
plt.show()

# Error Analysis trên tập Test
tn, fp, fn, tp = confusion_matrix(y_test, y_pred_scratch, labels=[0, 1]).ravel()
err_df = pd.DataFrame([
    {"Nhóm dự đoán": "True Negative (TN — Khách tốt đoán đúng)", "Số lượng": int(tn)},
    {"Nhóm dự đoán": "False Positive (FP — Cảnh báo nhầm khách tốt)", "Số lượng": int(fp)},
    {"Nhóm dự đoán": "False Negative (FN — Bỏ sót khách vỡ nợ)", "Số lượng": int(fn)},
    {"Nhóm dự đoán": "True Positive (TP — Bắt đúng khách vỡ nợ)", "Số lượng": int(tp)},
])
display(err_df)
print(f"Tỷ lệ phát hiện khách vỡ nợ (Recall): {tp / (tp + fn):.2%} ({tp:,}/{tp + fn:,})")
print(f"Tỷ lệ bỏ sót khách vỡ nợ (FNR)     : {fn / (tp + fn):.2%} ({fn:,}/{tp + fn:,})")

## 7. Quy trình kiểm tra & nộp bài trên nhánh `feature/member3-lightgbm`

Sau khi tải 6 file CSV Home Credit vào `data/` và nhấn **Run All** cho notebook này:
1. Kiểm tra file kết quả đã sinh tại `results/member3_lightgbm.json`.
2. Chạy bảng so sánh tổng hợp toàn nhóm:
   ```powershell
   python compare_models.py
   ```
3. Chỉ stage & push đúng 2 file riêng của Thành viên 3 (tuyệt đối không sửa `src/` hay commit `data/*.csv`):
   ```powershell
   git status
   git add member3_lightgbm.ipynb results/member3_lightgbm.json
   git commit -m "feat(member3): add LightGBM from scratch (Histogram + Leaf-wise + GOSS + EFB)"
   git push -u origin feature/member3-lightgbm
   ```
4. Mở **Pull Request** từ `feature/member3-lightgbm` vào `main` trên GitHub.